# PSE-823: Advanced Process Dynamics and Control
## Lecture 2 -- Modeling Review and Partial-Fraction Inversion
### (Coughanowr & LeBlanc, Ch. 2-3)

Date: __________

## Agenda

**Part A -- Modeling Review Sprint** (Ch. 2)
1. The mixing scenario: mass balance to first-order ODE
2. The energy balance analog
3. The Laplace transform and the 3-step procedure

**Part B -- Inversion by Partial Fractions** (Ch. 3)
4. The cover-up method
5. The mixing scenario, inverted
6. Complex and repeated roots
7. Qualitative behavior from root locations

Python runs throughout both Parts via `sympy`.

## Part A
### Modeling Review Sprint

(Coughanowr & LeBlanc, Ch. 2)
<!-- source: coughanowr-ch2 -->

### Concept

Recap at speed -- mostly CHE-323 territory. **The mixing scenario (Sec 2.1):** two streams mix, feed a heated tank, then a reactor. At 3:00 PM an operator swaps the flow rates.

**Figure 2-1 -- process flow diagram**

![](images/fig2-1_mixing_process_diagram.jpeg)

### Example

Before: $v_1=10$, $v_2=20$ L/min, $C_{a1}=1$, $C_{a2}=4$ g/L -> $C_{a3}=3$ g/L. After swap: $C_{a3}=2$ g/L. Total flow unchanged (30 L/min) -- only the *mix* changed.

$$5\frac{dC_a}{dt}+C_a=2,\quad C_a(0)=3 \;\Rightarrow\; C_a(t)=2+e^{-t/5}\ \text{(Eq. 2.2)}$$

![](images/fig2-2_mixing_transient.jpeg)

### Concept

Same logic, energy balance: stream 1 at $25^\circ C$, stream 2 at $55^\circ C$ -> $T_3=45^\circ C$; heater brings vessel to $80^\circ C$. After swap, $T_3\to35^\circ C$.

$$5\frac{dT}{dt}+T=70,\quad T(0)=80 \;\Rightarrow\; T(t)=70+10e^{-t/5}\ \text{(Eq. 2.4)}$$

![](images/fig2-3_energy_balance_a.jpeg)
![](images/fig2-4_energy_balance_b.jpeg)

### Concept

$$F(s)=\mathcal{L}\{f(t)\}=\int_0^\infty f(t)e^{-st}dt,\quad \mathcal{L}\{f'\}=sF(s)-f(0)$$

Linear. Table 2.1 collects standard transforms. Three-step procedure: (1) transform -- ICs enter here, (2) solve algebraically, (3) **invert** -- the hard step, today's real new material.

### Example

Transforming the mass balance (Eq. 2.10):

$$C_a(s)=\frac{2}{s(5s+1)}+\frac{15}{5s+1}$$

Same procedure, energy balance (Eq. 2.11):

$$T(s)=\frac{70}{s(5s+1)}+\frac{400}{5s+1}$$

Both now algebraic in $s$ -- Ch.3's job is inverting them.

### Python

The book uses MATLAB (`syms`, `laplace`, `dsolve`) throughout Ch.2-3. We use `sympy` -- same ideas, same three-step procedure.

In [1]:
import sympy as sp

t, s = sp.symbols('t s', positive=True)
Ca = sp.Function('C_a')
ode = sp.Eq(5 * Ca(t).diff(t) + Ca(t), 2)
sol = sp.dsolve(ode, Ca(t), ics={Ca(0): 3})
print(sol)  # should match C_a(t) = 2 + e^(-t/5)

Ca_s = sp.laplace_transform(sol.rhs, t, s, noconds=True)
print(sp.simplify(Ca_s))

Eq(C_a(t), 2 + exp(-t/5))
(15*s + 2)/(s*(5*s + 1))


### Your Turn (10 min)

**Problem 2.9.** At 3:00 PM the operator instead increases stream 1's flow to $20$ L/min; stream 2 ($20$ L/min, $4$ g/L) and stream 1's concentration ($1$ g/L) stay the same. Original $C_a(0)=3$ g/L.

1. Find the new $v_3$ and new steady-state $C_{a3}$.
2. Find the new $\tau$.
3. Transform the ODE and solve for $C_a(s)$. Do **not** invert yet -- that's Part B.

v3_new = 20+20 = 40 L/min (v3 DOES change this time, unlike the book's swap).
Ca3_new = (20*1+20*4)/40 = 100/40 = 2.5 g/L.
tau_new = V/v3 = 150/40 = 3.75 min.
3.75[sCa(s)-3] + Ca(s) = 2.5/s
Ca(s) = 2.5/[s(3.75s+1)] + 11.25/(3.75s+1)
Common gap: reusing tau=5 from the book's example without noticing v3 changed this time.

### Check

Cold-call for the new $\tau$. Why did $\tau$ change this time, when it didn't in the book's own example?

Because v3 itself changed here -- only stream 1's flow increased, unlike the book's swap where v1+v2 stayed at 30.

## Part B
### Inversion by Partial Fractions

(Coughanowr & LeBlanc, Ch. 3)
<!-- source: coughanowr-ch3 -->

### Concept

**Example 3.1 (basic case).** Solve $\frac{dx}{dt}+x=1$, $x(0)=0$: $x(s)=\frac{1}{s(s+1)}=\frac{A}{s}+\frac{B}{s+1}$.

**Heaviside cover-up**: cover $s$, set $s=0$: $A=1$. Cover $(s+1)$, set $s=-1$: $B=-1$.

$$x(t)=1-e^{-t}$$

### Example -- Example 3.2, the mixing scenario, inverted

Applying cover-up to Part A's Eq. 2.10:

$$\frac{2}{s(5s+1)}=\frac{2}{s}-\frac{2}{s+1/5} \;\Rightarrow\; C_a(s)=\frac{2}{s}+\frac{1}{s+1/5}$$

$$C_a(t)=2+e^{-t/5}$$

-- matching Part A's direct separation-of-variables result. Same method inverts Eq. 2.11 to $T(t)=70+10e^{-t/5}$.

### Concept -- complex conjugate roots

**Example 3.4.** $\frac{d^2x}{dt^2}+2\frac{dx}{dt}+2x=2$, zero ICs: $x(s)=\frac{2}{s(s^2+2s+2)}$. Roots $-1\pm j$ -- keep the quadratic unfactored, complete the square:

$$x(s)=\frac{1}{s}-\frac{(s+1)+1}{(s+1)^2+1^2} \;\Rightarrow\; x(t)=1-e^{-t}(\cos t+\sin t)$$

### Concept -- repeated roots

**Example 3.5.** $\frac{d^3x}{dt^3}+3\frac{d^2x}{dt^2}+3\frac{dx}{dt}+x=1$, zero ICs: $x(s)=\frac{1}{s(s+1)^3}$.

Cover-up gives $A=1$, $B=-1$ directly; $C,D$ need coefficient matching: $C=-1$, $D=-1$.

$$x(t)=1-e^{-t}\left(\frac{t^2}{2}+t+1\right)$$

### Concept -- qualitative behavior from root locations (Sec 3.2)

The **form** of $x(t)$ is readable directly from where the roots sit in the complex plane.

![](images/fig3-1_root_locations.jpeg)

Roots left of the imaginary axis -> decay. Right -> **growth** (instability -- Lecture 1's warning, formalized). On the axis -> sustained oscillation, or growing power series if repeated.

### Python

MATLAB has no dedicated partial-fractions command -- the book's workaround is `diff(int(x))`. `sympy.apart` does it directly.

In [2]:
import sympy as sp

s = sp.symbols('s')
expr = 1 / (s * (s + 1))
print(sp.apart(expr))              # matches the hand cover-up: 1/s - 1/(s+1)

t = sp.symbols('t', positive=True)
x_t = sp.inverse_laplace_transform(expr, s, t)
print(x_t)                          # 1 - exp(-t)

-1/(s + 1) + 1/s
1 - exp(-t)


### Your Turn (12 min)

Invert the s-domain expression from Part A's Your Turn (Problem 2.9):

$$C_a(s)=\frac{2.5}{s(3.75s+1)}+\frac{11.25}{3.75s+1}$$

Identify the root structure, apply cover-up, invert fully to get $C_a(t)$.

Rewrite 3.75s+1 = 3.75(s+0.2667).
First term: (2.5/3.75)/[s(s+0.2667)] = A/s + B/(s+0.2667), cover-up -> A=2.5, B=-2.5.
Second term: 11.25/[3.75(s+0.2667)] = 3/(s+0.2667).
Sum: 2.5/s - 2.5/(s+0.2667) + 3/(s+0.2667) = 2.5/s + 0.5/(s+0.2667)
Ca(t) = 2.5 + 0.5 e^{-0.2667t} = 2.5 + 0.5 e^{-t/3.75}
Self-check: at t=0, 2.5+0.5=3; as t->inf, ->2.5. Common slip: forgetting to convert 3.75s+1 to (s+0.2667) form before cover-up.

### Check

Cold-call for the root structure before inverting. How do you know your final answer is right, without redoing the algebra?

It should match Ca(0)=3 and the Part A steady-state target of 2.5 -- a sanity check, not a coincidence.

### Wrap-up

**Covered today:**
- The mixing scenario's mass/energy balances to first-order ODEs; Laplace definition and linearity
- Inversion by partial fractions: cover-up, complex roots, repeated roots, qualitative behavior from root locations
- `sympy` throughout -- `dsolve`, `laplace_transform`, `apart`, `inverse_laplace_transform`

**Next class:** Chapter 7 -- second-order systems and transportation lag. The complex-root case just derived becomes the normal one.